In [ ]:
# ============================================
# Brasil Olímpico - Exportar SSMS -> JSON do site
# Roda no Google Colab
# ============================================
# Passo 1: No SSMS, clique com botão direito no banco jogosolimpicos
#   Tarefas > Gerar Scripts > Tabelas + Dados > Salvar como .sql
# Passo 2: No Colab, suba esse .sql (botão de upload no painel esquerdo)
# Passo 3: Rode as células abaixo
# Passo 4: Baixe o dados_brasil.json gerado e suba no GitHub

In [ ]:
from google.colab import files
import re, json, collections

print("Selecione o arquivo .sql gerado pelo SSMS")
uploaded = files.upload()  # vai abrir a janela de upload
nome_arquivo = list(uploaded.keys())[0]
print(f"Arquivo recebido: {nome_arquivo}")

Selecione o arquivo .sql gerado pelo SSMS


Saving dump_jogosolimpicos.sql to dump_jogosolimpicos (1).sql
Arquivo recebido: dump_jogosolimpicos (1).sql


In [ ]:
# O SSMS gera em UTF-16. Tente UTF-16, se falhar, tente UTF-8.
raw = open(nome_arquivo, 'rb').read()
try:
    t = raw.decode('utf-16')
except:
    t = raw.decode('utf-8', errors='ignore')

lines = [l for l in t.splitlines() if l.startswith("INSERT [jogosolimpicos].")]

def parse_values(line):
    m = re.search(r'VALUES \((.*)\)\s*$', line)
    s = m.group(1)
    parts = []; cur = ""; in_str = False; i = 0
    while i < len(s):
        ch = s[i]
        if ch == "'":
            if i+1 < len(s) and s[i+1] == "'":
                cur += "''"; i += 2; continue
            in_str = not in_str; cur += ch; i += 1; continue
        if ch == "," and not in_str:
            parts.append(cur.strip()); cur = ""; i += 1; continue
        cur += ch; i += 1
    parts.append(cur.strip())
    def clean(v):
        v = v.strip()
        if v == "NULL": return None
        mm = re.match(r"CAST\(N'(.*)' AS Date\)", v)
        if mm: return mm.group(1).replace("''", "'")
        if v.startswith("N'") and v.endswith("'"): return v[2:-1].replace("''", "'")
        try: return int(v)
        except: return v
    return [clean(x) for x in parts]

atletas = {}   # id -> dict
edicoes = {}   # id -> dict
medalhas_map = {}  # id -> dict
rel = []       # lista de [ID, MedalhaID, AtletaID, EdicaoID]

for l in lines:
    if ".[atleta]" in l:
        r = parse_values(l)
        # [AtletaID, Nome_Principal, Nome_Complemento, Sexo, Data_Nascimento, Data_Morte, Cidade, Estado, Pais]
        atletas[r[0]] = {"id": r[0], "nome": r[1], "nome_oficial": r[2], "sexo": r[3],
                         "nasc": r[4], "cidade": r[6], "estado": r[7], "pais": r[8]}
    elif ".[edicao]" in l:
        r = parse_values(l)
        # [EdicaoID, Cidade, Pais, Ano, Verao_Inverno, Abertura, Encerramento, Inicio, Fim]
        edicoes[r[0]] = {"id": r[0], "cidade": r[1], "pais": r[2], "ano": r[3], "estacao": r[4]}
    elif ".[medalha]" in l:
        r = parse_values(l)
        # [MedalhaID, EdicaoID, Data, Medalha, Individual_Coletivo, Modalidade, Evento, Genero]
        medalhas_map[r[0]] = {"id": r[0], "edicao_id": r[1], "data": r[2], "tipo": r[3],
                              "indiv": r[4], "modalidade": r[5], "evento": r[6], "genero": r[7]}
    elif ".[medalhascomatletas]" in l:
        rel.append(parse_values(l))

print(f"atletas: {len(atletas)} | edicoes: {len(edicoes)} | medalhas: {len(medalhas_map)} | vínculos: {len(rel)}")

atletas: 464 | edicoes: 55 | medalhas: 171 | vínculos: 611


In [ ]:
# Se alguma falhar, pare aqui e confira o banco antes de gerar o JSON
assert len(atletas) == len(set(atletas.keys())), "ID de atleta duplicado"
assert len(medalhas_map) == len(set(medalhas_map.keys())), "ID de medalha duplicado"
atleta_ids = set(atletas.keys()); edicao_ids = set(edicoes.keys()); medalha_ids = set(medalhas_map.keys())
assert all(m["edicao_id"] in edicao_ids for m in medalhas_map.values()), "Medalha sem edição"
assert all(r[1] in medalha_ids and r[2] in atleta_ids for r in rel), "Vínculo com FK quebrada"
assert len(set((r[1], r[2]) for r in rel)) == len(rel), "Vínculo duplicado (medalha+atleta)"
print("✅ Validações OK - sem FK quebrada, sem duplicado")

# Totais por tipo
from collections import Counter
print("Totais:", Counter(m["tipo"] for m in medalhas_map.values()))

✅ Validações OK - sem FK quebrada, sem duplicado
Totais: Counter({'Bronze': 81, 'Prata': 49, 'Ouro': 41})


In [ ]:
# Mapa medalha -> atletas
medalha_para_atletas = collections.defaultdict(list)
for r in rel:
    medalha_para_atletas[r[1]].append(atletas[r[2]])

def genero_label(g):
    return {"M": "Masculino", "F": "Feminino", "F&M": "Misto"}.get(g, g)

medalhas = []
for mid, m in sorted(medalhas_map.items()):
    ed = edicoes[m["edicao_id"]]
    ats = medalha_para_atletas[mid]
    det = [{"nome": a["nome"], "nome_oficial": a["nome_oficial"],
            "busca": (a["nome"] + " " + (a["nome_oficial"] or "")).strip()} for a in ats]
    medalhas.append({
        "id": mid, "ano": ed["ano"], "cidade": ed["cidade"], "estacao": ed["estacao"],
        "modalidade": m["modalidade"], "evento": m["evento"], "genero": m["genero"],
        "genero_label": genero_label(m["genero"]), "indiv": m["indiv"], "data": m["data"],
        "medalha_tipo": m["tipo"], "atletas": [a["nome"] for a in ats],
        "atletas_detalhe": det,
        "busca_atletas": " ".join(d["busca"] for d in det)
    })

# by_edicao: TODAS as 55 edições, incluindo zeros
by_edicao = []
for eid, e in sorted(edicoes.items(), key=lambda x: (x[1]["ano"], x[1]["estacao"])):
    c = Counter(m["tipo"] for m in medalhas_map.values() if m["edicao_id"] == eid)
    by_edicao.append({"ano": e["ano"], "cidade": e["cidade"], "estacao": e["estacao"],
                      "ouro": c.get("Ouro", 0), "prata": c.get("Prata", 0), "bronze": c.get("Bronze", 0),
                      "total": sum(c.values())})

# by_modalidade
by_mod = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for m in medalhas_map.values():
    k = m["modalidade"]; by_mod[k]["total"] += 1
    by_mod[k]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_modalidade = [{"modalidade_nome": k, "total_medalhas": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]}
                 for k, v in sorted(by_mod.items(), key=lambda x: -x[1]["total"])]

# by_estado: conta PARTICIPAÇÕES (1 medalha coletiva conta 1x por atleta) - o site exibe como "medalhas"
by_est = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for r in rel:
    a = atletas[r[2]]; m = medalhas_map[r[1]]
    if a["pais"] != "Brasil": continue
    est = a["estado"] or "Não informado"
    by_est[est]["total"] += 1
    by_est[est]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_estado = [{"estado_nome": k, "total_participacoes": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]}
             for k, v in sorted(by_est.items(), key=lambda x: -x[1]["total"])]

# by_atleta + ranking
by_atl = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for r in rel:
    m = medalhas_map[r[1]]
    by_atl[r[2]]["total"] += 1
    by_atl[r[2]]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_atleta = []
ranking = []
for aid, v in sorted(by_atl.items(), key=lambda x: (-x[1]["total"], -x[1]["ouro"], -x[1]["prata"])):
    a = atletas[aid]
    by_atleta.append({"atleta_nome": a["nome"], "total_medalhas": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]})
    ranking.append({"nome": a["nome"], "sexo": a["sexo"], "estado": a["estado"], "pais": a["pais"], "cidade": a["cidade"],
                    "total": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"],
                    "nome_oficial": a["nome_oficial"], "busca": (a["nome"] + " " + (a["nome_oficial"] or "")).strip()})

# by_genero por edição (com zeros)
by_ge = collections.defaultdict(lambda: {"Masculino":0,"Feminino":0,"Misto":0})
for m in medalhas:
    by_ge[(m["ano"], m["estacao"])][m["genero_label"]] += 1
by_genero_edicao_completa = []
for e in by_edicao:
    v = by_ge.get((e["ano"], e["estacao"]), {"Masculino":0,"Feminino":0,"Misto":0})
    by_genero_edicao_completa.append({"ano": e["ano"], "cidade": e["cidade"], "estacao": e["estacao"],
                                      "Masculino": v["Masculino"], "Feminino": v["Feminino"], "Misto": v["Misto"],
                                      "total": sum(v.values())})
# Compatibilidade: by_genero_ano (agrupado por ano)
by_genero_ano = []
for ano in sorted(set(m["ano"] for m in medalhas)):
    agg = {"Masculino":0,"Feminino":0,"Misto":0}
    for m in medalhas:
        if m["ano"] == ano: agg[m["genero_label"]] += 1
    by_genero_ano.append({"ano": ano, **agg})

dados = {"medalhas": medalhas, "by_edicao": by_edicao, "by_modalidade": by_modalidade,
         "by_estado": by_estado, "by_atleta": by_atleta, "ranking": ranking,
         "by_genero_edicao_completa": by_genero_edicao_completa, "by_genero_ano": by_genero_ano,
         "edicoes": by_edicao, "by_edicao_completa": by_edicao}

with open("dados_brasil.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, ensure_ascii=False, indent=2)

print(f"✅ Gerado dados_brasil.json: {len(medalhas)} medalhas, {len(by_edicao)} edições, {len(ranking)} medalhistas")
print(f"   Soma modalidades: {sum(x['total_medalhas'] for x in by_modalidade)} (deve ser = medalhas)")
print(f"   Top atleta: {ranking[0]['nome']} com {ranking[0]['total']} medalhas")

✅ Gerado dados_brasil.json: 171 medalhas, 55 edições, 464 medalhistas
   Soma modalidades: 171 (deve ser = medalhas)
   Top atleta: Rebeca Andrade com 6 medalhas


In [ ]:
files.download("dados_brasil.json")
print("Baixe o arquivo e suba no GitHub no lugar do antigo. Pronto!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Baixe o arquivo e suba no GitHub no lugar do antigo. Pronto!
